# Deploy a Managed Online Endpoint

Generate the scoring function and request, create a Microsoft Entra-authenticated endpoint, deploy the registered foundation model at zero traffic, invoke it directly, and optionally promote traffic.

**Source:** Adapted from [Azure/azureml-examples simple managed deployment](https://github.com/Azure/azureml-examples/blob/37c3572b3ceafdaaa90ee4503c920cfff899df1f/sdk/python/endpoints/online/managed/online-endpoints-simple-deployment.ipynb) and this repository's endpoint patterns, MIT License.

In [ ]:
from pathlib import Path
import ast
import json
import os
import textwrap

import jwt
from azure.ai.ml import MLClient
from azure.ai.ml.constants import ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
)
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
access_token = credential.get_token("https://management.azure.com/.default")
token_claims = jwt.decode(access_token.token, options={"verify_signature": False})
ORCHESTRATOR_OBJECT_ID = token_claims.get("oid", "<COMPUTE_INSTANCE_UMI_OBJECT_ID>")
ENDPOINT_NAME = os.environ["WORKSHOP_ENDPOINT_NAME"]
DEPLOYMENT_NAME = os.environ["WORKSHOP_DEPLOYMENT_NAME"]
MODEL_NAME = os.environ["WORKSHOP_MODEL_NAME"]
MODEL_VERSION = os.environ["WORKSHOP_MODEL_VERSION"]
ENVIRONMENT_NAME = os.environ["WORKSHOP_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["WORKSHOP_ENVIRONMENT_VERSION"]
INSTANCE_TYPE = os.environ["AZUREML_ONLINE_INSTANCE_TYPE"]
PUBLIC_ACCESS = os.getenv("AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS", "disabled")
IDENTITY_ID = os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip()
DEPLOY = os.getenv("DEPLOY_FOUNDATION_ENDPOINT", "false").lower() in {"1", "true", "yes"}
PROMOTE = os.getenv("PROMOTE_FOUNDATION_TRAFFIC", "false").lower() in {"1", "true", "yes"}

In [ ]:
generated_dir = WORKSHOP_ROOT / "outputs/generated/foundations/online"
generated_code_dir = generated_dir / "code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
score_path = generated_code_dir / "score.py"
request_path = generated_dir / "request.json"
score_source = r'''
import json
import os
from pathlib import Path

import pandas as pd

_model = None


def init():
    global _model
    model_root = Path(os.environ["AZUREML_MODEL_DIR"])
    matches = list(model_root.rglob("model.json"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one model.json, found {len(matches)}")
    _model = json.loads(matches[0].read_text(encoding="utf-8"))


def run(raw_data):
    payload = json.loads(raw_data) if isinstance(raw_data, (str, bytes)) else raw_data
    input_data = payload.get("input_data", {})
    columns = input_data.get("columns")
    rows = input_data.get("data")
    if columns != _model["features"]:
        raise ValueError(f"Expected columns in this order: {_model['features']}")
    if not isinstance(rows, list) or not rows:
        raise ValueError("Request must contain at least one row")
    frame = pd.DataFrame(rows, columns=columns).apply(pd.to_numeric, errors="raise")
    predictions = _model["intercept"]
    for feature, coefficient in zip(
        _model["features"], _model["coefficients"], strict=True
    ):
        predictions = predictions + frame[feature] * coefficient
    return {"predictions": predictions.astype(float).tolist()}
'''
score_source = textwrap.dedent(score_source).lstrip()
ast.parse(score_source, filename=str(score_path))
score_path.write_text(score_source, encoding="utf-8")
request = {
    "input_data": {
        "columns": ["tripDistance", "passengerCount"],
        "data": [[2.5, 1], [7.0, 2]],
    }
}
request_path.write_text(json.dumps(request, indent=2) + "\n", encoding="utf-8")
print(f"Generated scoring script: {score_path}")
print(f"Generated request: {request_path}")

EXPECTED_BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"

registered_model = ml_client.models.get(MODEL_NAME, MODEL_VERSION)
registered_environment = ml_client.environments.get(ENVIRONMENT_NAME, ENVIRONMENT_VERSION)
if registered_environment.image != EXPECTED_BASE_IMAGE:
    raise ValueError(
        f"Environment {ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION} uses "
        f"{registered_environment.image}, expected {EXPECTED_BASE_IMAGE}. "
        "Rerun 03_register_environment.ipynb with a new version."
    )
print(f"Model: {registered_model.name}:{registered_model.version}")
print(f"Environment: {registered_environment.name}:{registered_environment.version}")
print(f"Base image: {registered_environment.image}")

identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[ManagedIdentityConfiguration(resource_id=IDENTITY_ID)],
    )

endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Azure ML workshop foundation endpoint",
    auth_mode="aad_token",
    identity=identity,
    public_network_access=PUBLIC_ACCESS,
    tags={"workshop": "azureml-h2o", "purpose": "foundations"},
)
deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{MODEL_NAME}:{MODEL_VERSION}",
    environment=f"azureml:{ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}",
    code_configuration=CodeConfiguration(
        code=str(generated_code_dir),
        scoring_script="score.py",
    ),
    instance_type=INSTANCE_TYPE,
    instance_count=1,
)

if DEPLOY:
    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        print(f"Using existing endpoint: {endpoint.name}")
    except ResourceNotFoundError:
        try:
            endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
        except HttpResponseError as error:
            if error.status_code == 403 and IDENTITY_ID:
                raise PermissionError(
                    "The compute-instance UMI can create endpoints but cannot assign the configured "
                    "endpoint UMI. An Azure RBAC administrator must run:\n"
                    f"az role assignment create --assignee-object-id {ORCHESTRATOR_OBJECT_ID} "
                    "--assignee-principal-type ServicePrincipal "
                    "--role 'Managed Identity Operator' "
                    f"--scope '{IDENTITY_ID}'\n"
                    "After role propagation, rerun Cells 2-3."
                ) from error
            raise
        print(f"Created endpoint: {endpoint.name}")

    deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
    if deployment.provisioning_state != "Succeeded":
        raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")

    response = ml_client.online_endpoints.invoke(
        endpoint_name=ENDPOINT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        request_file=str(request_path),
    )
    body = json.loads(response)
    assert body["predictions"] == [9.2, 20.2]
    print(body)

    if PROMOTE:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        endpoint.traffic = {DEPLOYMENT_NAME: 100}
        ml_client.online_endpoints.begin_create_or_update(endpoint).result()
        print(f"Traffic promoted to {DEPLOYMENT_NAME}")
    else:
        print("Traffic remains unchanged. Set PROMOTE_FOUNDATION_TRAFFIC=true to promote.")
else:
    print(f"Prepared endpoint/deployment: {ENDPOINT_NAME}/{DEPLOYMENT_NAME}")
    print("Deployment disabled. Set DEPLOY_FOUNDATION_ENDPOINT=true in workshop/.env.")

## Expected Result

The deployment reaches `Succeeded`, direct invocation returns `[9.2, 20.2]`, and endpoint traffic changes only when the promotion switch is enabled.

Next: `../02_jobs_and_pipelines/01_submit_single_step_merge.ipynb`.